In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("final-dataset.csv", dtype=str)

chicken = df.iloc[1:, 3].tolist()
fetus = df.iloc[1:, 4].tolist()
states = df.iloc[1:, 5].tolist()
breastcancer = df.iloc[1:, 6].tolist()

In [2]:
import csv
import time
import random
import urllib.parse
from datetime import datetime

import tldextract
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── Config ────────────────────────────────────────────────────────────────────

OUTPUT_CSV  = "google_states_results.csv"
MAX_RESULTS = 10
DELAY       = 5.0

# ── Load queries ──────────────────────────────────────────────────────────────

df = pd.read_csv("final-dataset.csv", dtype=str)
queries = [q.strip() for q in states if isinstance(q, str) and q.strip()]

print(f"Loaded {len(queries)} states queries:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q!r}")

# ── Browser ───────────────────────────────────────────────────────────────────

def make_driver():
    opts = Options()
    opts.add_argument("--incognito")
    opts.add_argument("--no-first-run")
    opts.add_argument("--no-default-browser-check")
    opts.add_argument("--disable-extensions")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1280,900")
    opts.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    driver  = webdriver.Chrome(service=service, options=opts)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    return driver

def build_url(query: str) -> str:
    params = {"q": query, "hl": "en", "gl": "us", "num": "10", "udm": "14", "pws": "0"}
    return "https://www.google.com/search?" + urllib.parse.urlencode(params)

def extract_domain(url: str) -> str:
    """Full netloc minus www, e.g. cooking.nytimes.com"""
    try:
        return urllib.parse.urlparse(url).netloc.replace("www.", "").strip()
    except Exception:
        return ""

def extract_registrable_domain(url: str) -> str:
    """eTLD+1 only, e.g. nytimes.com — handles co.uk, com.au etc correctly"""
    try:
        ext = tldextract.extract(url)
        return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain
    except Exception:
        return ""

# ── Scraper — updated for current Google DOM ─────────────────────────────────

def scrape_page(driver, max_results=10) -> list[dict]:
    """
    tF2Cxc  = current Google organic result container (confirmed in your DOM)
    VwiC3b  = snippet div (also confirmed in your DOM)
    """
    wait = WebDriverWait(driver, 10)
    try:
        # Wait for tF2Cxc — the confirmed current container class
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.tF2Cxc")))
    except Exception:
        print("  WARNING: timed out waiting for div.tF2Cxc")
        # Print what's on the page to help debug further
        print("  Page title:", driver.title)
        return []

    containers = driver.find_elements(By.CSS_SELECTOR, "div.tF2Cxc")
    print(f"  Found {len(containers)} result containers (div.tF2Cxc)")

    results   = []
    seen_urls = set()
    rank      = 0

    for container in containers:
        if rank >= max_results:
            break

        # ── Title ─────────────────────────────────────────────────────────
        title = ""
        for sel in ["h3", "h3.LC20lb"]:
            try:
                title = container.find_element(By.CSS_SELECTOR, sel).text.strip()
                if title:
                    break
            except Exception:
                continue
        if not title:
            continue

        # ── URL ───────────────────────────────────────────────────────────
        url = ""
        try:
            for a in container.find_elements(By.CSS_SELECTOR, "a[href]"):
                href = a.get_attribute("href") or ""
                if href.startswith("http") and "google.com" not in href:
                    url = href
                    break
        except Exception:
            pass
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)

        domain              = extract_domain(url)
        registrable_domain  = extract_registrable_domain(url)

        # ── Snippet ───────────────────────────────────────────────────────
        snippet = ""
        for sel in [
            "div.VwiC3b",          # confirmed in your DOM
            "div.VwiC3b span",
            "span.aCOpRe",
            "div.IsZvec",
        ]:
            try:
                snippet = container.find_element(By.CSS_SELECTOR, sel).text.strip()
                if snippet:
                    break
            except Exception:
                continue

        rank += 1
        results.append({
            "rank":                rank,
            "title":               title,
            "url":                 url,
            "domain":              domain,
            "registrable_domain":  registrable_domain,
            "snippet":             snippet,
        })
        print(f"    [{rank:>2}] {registrable_domain:<30} {title[:50]}")

    return results

# ── CSV setup ─────────────────────────────────────────────────────────────────

fields = [
    "query_index", "col", "query",
    "rank", "title", "url", "domain", "registrable_domain", "snippet",
    "n_results", "timestamp",
]

# ── Main loop ─────────────────────────────────────────────────────────────────

driver = make_driver()

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    for i, query in enumerate(queries):
        print(f"\n{'='*60}")
        print(f"Query {i+1}/{len(queries)}: {query!r}")

        try:
            driver.get(build_url(query))
            time.sleep(random.uniform(2.5, 4.0))

            results   = scrape_page(driver, max_results=MAX_RESULTS)
            timestamp = datetime.utcnow().isoformat()

            if results:
                for r in results:
                    writer.writerow({
                        "query_index": i + 1,
                        "col":         "NPC",
                        "query":       query,
                        **r,
                        "n_results":   len(results),
                        "timestamp":   timestamp,
                    })
                print(f"  → {len(results)} results saved")
            else:
                writer.writerow({
                    "query_index": i+1, "col": "NPC", "query": query,
                    "rank": None, "title": "", "url": "", "domain": "",
                    "snippet": "", "n_results": 0, "timestamp": timestamp,
                })
                print("  → 0 results")

            f.flush()

        except Exception as e:
            print(f"  ERROR: {e}")
            writer.writerow({
                "query_index": i+1, "col": "NPC", "query": query,
                "rank": None, "title": f"ERROR: {e}", "url": "",
                "domain": "", "snippet": "", "n_results": 0,
                "timestamp": datetime.utcnow().isoformat(),
            })
            f.flush()

        if i < len(queries) - 1:
            sleep = DELAY + random.uniform(0.5, 2.0)
            print(f"  Sleeping {sleep:.1f}s...")
            time.sleep(sleep)

driver.quit()
print(f"\nDone. Results saved to: {OUTPUT_CSV}")

Loaded 227 states queries:
  1. 'abortion by state'
  2. 'v'
  3. 'How many states is abortion illegal'
  4. 'different state laws of abortion'
  5. 'how many states have outlawed abortion'
  6. 'how many states banned abortion'
  7. 'states that have outlawed abortion'
  8. 'states that have outlawed abortion'
  9. 'Which States outlawed abortion.'
  10. 's'
  11. 'abortion states'
  12. 'v'
  13. 'How many states have outlawed abortion'
  14. 'how many states have outlawed abortion'
  15. 'abortion outlawed'
  16. 'How many states banned abortion'
  17. 'how may states have outlawed abortion'
  18. 'states banning abortion'
  19. 'outlawed abortion rights by state'
  20. 'states who outlawed abortion'
  21. 'illegal abortion states'
  22. 'states outlawed abortion'
  23. 'where is abortion banned'
  24. 'in what states is abortion legal'
  25. 'how many states have outlawed abortion'
  26. 'How many states have outlawed abortion'
  27. 'How many states have outlawed abortion so far.'

In [ ]:
df_states = pd.read_csv("google_states_results.csv")
df_states

In [14]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# 1. The Matrix (Coefficients for your 8 inequalities)
A = np.array([
    [0, 0, 1, 1, 1, 1, 0, 1],
    [1, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 1, 1, 0, 0, 0, 1],
    [0, 0, 0, 0, 1, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 1],
    [0, 0, 0, 0, 0, 1, 1, 1]
])

# 2. The Right-Hand Side (your B vector)
b_upper = np.array([14, 9, 4, 10, 5, 4, 6, 12, 7])
# Since these are inequalities (<=), the lower bound is negative infinity
b_lower = np.full_like(b_upper, -np.inf)

# 3. Define Constraints: b_lower <= A @ x <= b_upper
constraints = LinearConstraint(A, b_lower, b_upper)

# 4. Objective Function: Maximize the sum (x1 + x2 + ... + x8)
# We use -1 to maximize because the solver minimizes by default
c = np.full(8, -1)

# 5. Variable Constraints: Must be non-negative (0 to infinity)
bounds = Bounds(0, np.inf)

# 6. Integer Requirement: Set 1 for variables that must be integers
integrality = np.ones(8) # All 8 variables must be integers

# Solve
res = milp(c=c, constraints=constraints, bounds=bounds, integrality=integrality)

if res.success:
    print("Optimization Successful!")
    variables = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']
    for var, val in zip(variables, res.x):
        print(f"{var} = {int(val)}") # Display as clean integers
    print(f"Total Max Value: {-res.fun:.2f}")
else:
    print("No integer solution found that satisfies all constraints.")

Optimization Successful!
x1 = 4
x2 = 0
x3 = 0
x4 = 0
x5 = 6
x6 = 2
x7 = 5
x8 = 0
Total Max Value: 17.00


/Users/ushnamalik/anaconda3/lib/python3.11/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')
